In [1]:
import numpy as np
import pandas as pd

In [2]:
# Path Config
TRAIN_PATH = r"C:\Datasets\train_SCkXwSx_cNdUk28_icK39vy_36Oa6WQ.csv"
TEST_PATH = r"C:\Datasets\test_opI5Wmb_N886n9d_eE68Ob0_dOXWldY.csv"
SAMPLE_PATH = r"C:\Datasets\sample_submission_ku1VMOX_Gd11zvJ_VxdIEsN_3crCTYo.csv"

In [3]:
# Load data
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

print( train.shape)
print( test.shape)

(89392, 12)
(59595, 11)


In [4]:
train.head()

,id,gender,area,qualification,income,marital_status,vintage,claim_amount,num_policies,policy,type_of_policy,cltv
0,1,Male,Urban,Bachelor,5L-10L,1,5,5790,More than 1,A,Platinum,64308
1,2,Male,Rural,High School,5L-10L,0,8,5080,More than 1,A,Platinum,515400
2,3,Male,Urban,Bachelor,5L-10L,1,8,2599,More than 1,A,Platinum,64212
3,4,Female,Rural,High School,5L-10L,0,7,0,More than 1,A,Platinum,97920
4,5,Male,Urban,High School,More than 10L,1,6,3508,More than 1,A,Gold,59736


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89392 entries, 0 to 89391
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              89392 non-null  int64 
 1   gender          89392 non-null  object
 2   area            89392 non-null  object
 3   qualification   89392 non-null  object
 4   income          89392 non-null  object
 5   marital_status  89392 non-null  int64 
 6   vintage         89392 non-null  int64 
 7   claim_amount    89392 non-null  int64 
 8   num_policies    89392 non-null  object
 9   policy          89392 non-null  object
 10  type_of_policy  89392 non-null  object
 11  cltv            89392 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 8.2+ MB


In [6]:
train.describe()

,id,marital_status,vintage,claim_amount,cltv
count,89392.000000,89392.000000,89392.000000,89392.000000,89392.000000
mean,44696.500000,0.575488,4.595669,4351.502416,97952.828978
std,25805.391969,0.494272,2.290446,3262.359775,90613.814793
min,1.000000,0.000000,0.000000,0.000000,24828.000000
25%,22348.750000,0.000000,3.000000,2406.000000,52836.000000
50%,44696.500000,1.000000,5.000000,4089.000000,66396.000000
75%,67044.250000,1.000000,6.000000,6094.000000,103440.000000
max,89392.000000,1.000000,8.000000,31894.000000,724068.000000


In [7]:
train.isnull().sum()

id                0
gender            0
area              0
qualification     0
income            0
marital_status    0
vintage           0
claim_amount      0
num_policies      0
policy            0
type_of_policy    0
cltv              0
dtype: int64

#### So it an Supervised Learning --> Regression problem and where the `target variable (y = cltv)` is `cltv (Customer Lifetime value)` and it is an continuous numeric value and the `inpute fearure (X)` of an Tabular data

## Preprocessing
- Preparing data to unify format to combine and ready for feature engineering

In [8]:
# creating a separate copy of dataset to prevent modifying the original data
train = train.copy()  
test = test.copy()

In [9]:
# Adding a "flag column" to identify rows and after merging both datasets we can split them back
train["is_train"] = 1
test["is_train"] = 0

In [10]:
# Target column to test 
test["cltv"] = -1

In [11]:
# fixing 'num_policies' where modifing "More than 1" --> '2'
train["num_policies"] = train["num_policies"].replace({"More than 1": 2}).astype(int)
test["num_policies"] = test["num_policies"].replace({"More than 1": 2}).astype(int)

# combine the Train + Test
df = pd.concat([train, test], axis=0).reset_index(drop=True)

## Feature engneering

In [12]:
# converting to numeric
df["income"] = pd.to_numeric(df["income"], errors="coerce")
df["claim_amount"] = pd.to_numeric(df["claim_amount"], errors="coerce")

In [13]:
# removeing commas and fixing numbers
num_cols = ["income", "claim_amount", "num_policies", "vintage"]
for col in num_cols:
    df[col] = df[col].astype(str).str.replace(",", "", regex=False)
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [14]:
# Here we are using the log trasformation for "income" & "claims" are skewed so, log makes distribution more normal
df["log_income"] = np.log1p(df["income"])
df["log_claim"] = np.log1p(df["claim_amount"])

In [15]:
# Ratio 
df["claim_income_ratio"] = df["claim_amount"] / (df["income"] + 1)
#df["claim_per_policy"] = df["claim_amount"] / (df["num_policies"] + 1)
df["policies_per_year"] = df["num_policies"] / (df["vintage"] + 1)

df["income_per_policy"] = df["income"] / (df["num_policies"] + 1)
#df["claim_to_vintage"] = df["claim_amount"] / (df["vintage"] + 1)

In [16]:
df["claim_per_policy_log"] = np.log1p(df["claim_amount"] / (df["num_policies"] + 1))
df["policy_intensity"] = df["num_policies"] / (df["vintage"] + 1)
df["policy_intensity_log"] = np.log1p(df["policy_intensity"])
#df["claim_pressure"] = df["claim_amount"] / (df["vintage"] + 1)
#df["claim_pressure_log"] = np.log1p(df["claim_pressure"])
#df["income_claim_diff"] = df["income"] - df["claim_amount"]
#df["income_claim_ratio_log"] = np.log1p(df["income"] / (df["claim_amount"] + 1))

In [17]:
# Group based feature 
#df["area_income_mean"] = df.groupby("area")["income"].transform("mean")
#df["policy_claim_mean"] = df.groupby("policy")["claim_amount"].transform("mean")


# Encoding

In [18]:
# converting categorical columns into numbers 
cat_cols = ["gender", "area", "qualification", "policy", "type_of_policy"]
for col in cat_cols:
    df[col] = df[col].astype("category").cat.codes

## Split back 

In [19]:
# combineding dataset back into train and test 
train = df[df["is_train"] == 1].drop(columns=["is_train"],axis=1).copy()
test = df[df["is_train"] == 0].drop(columns=["is_train", "cltv"],axis=1).copy()


In [20]:
for col in ["area", "policy"]:
    means = train.groupby(col)["cltv"].mean()
    
    train[col+"_te"] = train[col].map(means)
    test[col+"_te"] = test[col].map(means)
    
    train[col+"_te"].fillna(train["cltv"].mean(), inplace=True)
    test[col+"_te"].fillna(train["cltv"].mean(), inplace=True)

C:\Users\dyaga\AppData\Local\Temp\ipykernel_24140\620833682.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train[col+"_te"].fillna(train["cltv"].mean(), inplace=True)
C:\Users\dyaga\AppData\Local\Temp\ipykernel_24140\620833682.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

## MODEL TRAINING
- Here we are using the LightGBM`(lgb.LGBMRegressor)` which is fast gradient boosting model from the best performance
- And KFold for cross validation where data is splits into 5(parts), train on 80% and valid on 20% each fold

In [21]:
import lightgbm as lgb
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

# Feature (X) , Target (y)
x = train.drop(columns=["cltv", "id"],axis=1) 
y = train["cltv"]

test = test.reindex(columns=x.columns, fill_value=0)

x = x.fillna(0)
test = test.fillna(0)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(x))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(kf.split(x), start=1):
    print(f"Fold {fold}")

    x_train, x_val = x.iloc[train_idx], x.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=5,
        num_leaves=25,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=0.5,
        reg_lambda=0.5,
        random_state=42
    )

    model.fit(
        x_train,
        y_train,
        eval_set=[(x_val, y_val)],
        eval_metric="r2",
        callbacks=[lgb.early_stopping(100)],
    )

    oof_preds[val_idx] = model.predict(x_val)
    test_preds += model.predict(test) / 5



Fold 1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001569 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 844
[LightGBM] [Info] Number of data points in the train set: 71513, number of used features: 16
[LightGBM] [Info] Start training from score 98194.843343
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

## Evaluation

In [22]:
score = r2_score(y, oof_preds)
print("Final R2 Score:", score)

Final R2 Score: 0.15944949473080783


## Submission 

In [23]:
# Cleaning predictions
test_preds = np.nan_to_num(test_preds, nan=0, posinf=0, neginf=0)

# Clip extreme values
test_preds = np.clip(test_preds, 0, 200000)

In [24]:
submission = sample.copy()
submission["cltv"] = test_preds
submission.to_csv("submission.csv", index=False)

print("Submission saved to submission.csv")

Submission saved to submission.csv


In [25]:
print(submission.head())

      id           cltv
0  89393   92764.536861
1  89394  124830.049050
2  89395   94999.275425
3  89396   88198.783335
4  89397  129915.742587


In [26]:
print(submission.isna().sum())

id      0
cltv    0
dtype: int64


In [27]:
submission.to_csv("submission.csv", index=False)